# STTN-CP — Spatial-Temporal Transformer with Contrastive Pretraining

**Credit card fraud detection | maintained open-source implementation**

This notebook runs the same maintained source implementation used by the command-line training and evaluation scripts. The spatial module attends across the original transaction features, the temporal module attends across the transaction window, and the classifier receives the complete window.


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import ConfusionMatrixDisplay

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.architecture import STTNCP, count_trainable_parameters
from src.data import (
    load_creditcard, chronological_split, scale_splits,
    build_labeled_windows, make_loader, class_weights,
)
from src.train import augment, predict_and_score

SEED = 42
SEQ_LEN = 8
EMBED_DIM = 128
N_BLOCKS = 3
N_HEADS = 4
MLP_DIM = 256
PROJ_DIM = 64
BATCH_SIZE = 128
EPOCHS = 50
LR = 1e-3
TEMPERATURE = 0.07
LAMBDA_CONTRASTIVE = 0.5
PATIENCE = 10
NOISE_STD = 0.02
DATA_PATH = ROOT / "data" / "creditcard.csv"
RUN_BENCHMARK = DATA_PATH.exists()

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Dataset present: {RUN_BENCHMARK} -> {DATA_PATH}")

Device: cpu
Dataset present: False -> /mnt/data/sttn-cp-ccfd-v2-clean/data/creditcard.csv


## 1. Dataset and partitioning

The default repository protocol sorts transactions by `Time`, creates 70% training, 10% validation, and 20% test partitions, fits the scaler on training data only, and builds windows independently within each partition.

In [2]:
if RUN_BENCHMARK:
    df = load_creditcard(DATA_PATH)
    print("Dataset shape:", df.shape)
    print("Class counts:")
    print(df["Class"].value_counts().sort_index())

    splits = chronological_split(df, train_fraction=0.70, validation_fraction=0.10)
    (X_train, y_train), (X_val, y_val), (X_test, y_test), scaler, features = scale_splits(
        splits.train, splits.validation, splits.test
    )
    print("Partitions:", len(X_train), len(X_val), len(X_test))
    print("Features:", len(features), features[:5], "...", features[-3:])

    X_train_w, y_train_w = build_labeled_windows(X_train, y_train, SEQ_LEN)
    X_val_w, y_val_w = build_labeled_windows(X_val, y_val, SEQ_LEN)
    X_test_w, y_test_w = build_labeled_windows(X_test, y_test, SEQ_LEN)
    print("Window shapes:", X_train_w.shape, X_val_w.shape, X_test_w.shape)
else:
    print("creditcard.csv is not present. The benchmark path is skipped; the software smoke test below remains executable.")

creditcard.csv is not present. The benchmark path is skipped; the software smoke test below remains executable.


## 2. Inspect the architecture

The key implementation detail is the direction of attention:

- **Spatial:** `D` original transaction features become tokens at each timestep.
- **Temporal:** the `T` transactions in the window become tokens after the input projection.
- **Residual flow:** spatial → residual → temporal → residual.

The classifier therefore consumes a complete `(batch, 8, features)` window.

In [3]:
model = STTNCP(
    input_dim=30,
    seq_len=SEQ_LEN,
    embed_dim=EMBED_DIM,
    n_blocks=N_BLOCKS,
    n_heads=N_HEADS,
    mlp_dim=MLP_DIM,
    proj_dim=PROJ_DIM,
).to(DEVICE)

print(model)
print(f"Trainable parameters: {count_trainable_parameters(model):,}")

with torch.no_grad():
    sample = torch.randn(4, SEQ_LEN, 30, device=DEVICE)
    print("Backbone:", model.forward_backbone(sample).shape)
    print("Logits:", model(sample).shape)

STTNCP(
  (input_embed): Linear(in_features=30, out_features=128, bias=True)
  (temporal_positional): PositionalEncoding()
  (st_blocks): ModuleList(
    (0-2): 3 x STBlock(
      (spatial): SpatialTransformerBlock(
        (feature_embed): Linear(in_features=1, out_features=128, bias=True)
        (feature_pos): PositionalEncoding()
        (transformer): TransformerEncoder(
          (layers): ModuleList(
            (0): TransformerEncoderLayer(
              (self_attn): MultiheadAttention(
                (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
              )
              (linear1): Linear(in_features=128, out_features=256, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
              (linear2): Linear(in_features=256, out_features=128, bias=True)
              (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
             

Logits: torch.Size([4, 2])


## 3. Training configuration

The maintained implementation uses 50 training epochs, batch size 128, Adam with learning rate 0.001, and InfoNCE temperature 0.07 by default. The combined objective is implemented as:

`L_total = L_classification + λ * L_contrastive`

Early stopping is selected using validation F1.

In [4]:
if RUN_BENCHMARK:
    train_loader = make_loader(X_train_w, y_train_w, BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = make_loader(X_val_w, y_val_w, BATCH_SIZE, shuffle=False)
    test_loader = make_loader(X_test_w, y_test_w, BATCH_SIZE, shuffle=False)

    model = STTNCP(
        input_dim=X_train.shape[1], seq_len=SEQ_LEN,
        embed_dim=EMBED_DIM, n_blocks=N_BLOCKS,
        n_heads=N_HEADS, mlp_dim=MLP_DIM, proj_dim=PROJ_DIM,
    ).to(DEVICE)
    weights = class_weights(y_train_w).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = []
    best_state, best_val_f1, stale = None, -1.0, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_values, class_values, contrast_values = [], [], []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            view1 = augment(xb, NOISE_STD)
            view2 = augment(xb, NOISE_STD)
            z1 = model.project(model.forward_backbone(view1))
            z2 = model.project(model.forward_backbone(view2))
            logits = model(xb)

            from src.architecture import combined_loss
            loss, class_loss_value, contrast_loss_value = combined_loss(
                logits, yb, z1, z2, criterion,
                lambda_contrastive=LAMBDA_CONTRASTIVE,
                temperature=TEMPERATURE,
            )
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_values.append(loss.item())
            class_values.append(class_loss_value.item())
            contrast_values.append(contrast_loss_value.item())

        scheduler.step()
        val_metrics = predict_and_score(model, val_loader, DEVICE)
        history.append({
            "epoch": epoch,
            "train_loss": float(np.mean(total_values)),
            "train_class_loss": float(np.mean(class_values)),
            "train_contrastive_loss": float(np.mean(contrast_values)),
            "val_f1": val_metrics["f1"],
        })
        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    test_metrics = predict_and_score(model, test_loader, DEVICE)
    print("Test metrics:")
    print(json.dumps(test_metrics, indent=2))
else:
    history = []
    test_metrics = None

## 4. Software smoke test

This section runs even when the Kaggle file is not available. It uses a small deterministic synthetic dataset only to exercise the complete model path. The numbers are **not benchmark results**.

In [5]:
def synthetic_windows(n=256, seq_len=8, input_dim=30, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n + seq_len - 1, input_dim)).astype(np.float32)
    y = (X[:, 0] + 0.2 * X[:, 1] > 0).astype(np.int64)
    return build_labeled_windows(X, y, seq_len)

sx, sy = synthetic_windows()
smoke_model = STTNCP(input_dim=30, seq_len=SEQ_LEN, embed_dim=32, n_blocks=1, n_heads=4, mlp_dim=64, proj_dim=16)
with torch.no_grad():
    smoke_logits = smoke_model(torch.from_numpy(sx[:16]))
    smoke_z = smoke_model.project(smoke_model.forward_backbone(torch.from_numpy(sx[:16])))

from src.architecture import infonce_loss
smoke_loss = infonce_loss(smoke_z[:8], smoke_z[8:16])
print("Synthetic windows:", sx.shape)
print("Classifier output:", smoke_logits.shape)
print("InfoNCE loss finite:", bool(torch.isfinite(smoke_loss)))

Synthetic windows: (256, 8, 30)
Classifier output: torch.Size([16, 2])
InfoNCE loss finite: True


## 5. Current implementation outputs

A dataset-backed run produces metrics from the current implementation. These values are measurements from that run and are not hard-coded benchmark references.

## 6. Running the current implementation

For a dataset-backed run, place `creditcard.csv` under `data/` and run the dataset and training sections. For a clean command-line run from the repository root:

```bash
python -m pytest -q
python -m src.train --data-path data/creditcard.csv --output-dir outputs
python -m src.evaluate --data-path data/creditcard.csv --checkpoint outputs/best_model.pt --output outputs/test_metrics.json
```

The notebook and CLI use the same source package, so both exercise the same maintained model implementation.